<a href="https://colab.research.google.com/github/supriyag123/PHD_Pub/blob/main/AGENTIC-MODULE6.3-Additional%20Analyses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MODULE 6.3 — ASPIRE ABLATION STUDY
# Standalone notebook — no X, no Transformer, no sensor models
# All results derived from PRECOMP[:2000] matching paper evaluation
# ============================================================

import numpy as np
import json
import os
from sklearn.metrics import precision_score, recall_score, f1_score
from typing import Optional, Dict, Any, List
from datetime import datetime

# ============================================================
# PATHS
# ============================================================
PRECOMP_PATH  = "/content/drive/MyDrive/PHD/2025/cache/holdout_precomp.json"
label_path    = "/content/drive/MyDrive/PHD/2025/TEMP_OUTPUT_METROPM/window_labels_3class.npy"
holdout_mask_path = "/content/drive/MyDrive/PHD/2025/TEMP_OUTPUT_METROPM/holdout_mask.npy"
ABLATION_PATH = "/content/drive/MyDrive/PHD/2025/ablation_results.json"

# ============================================================
# PAPER EVALUATION WINDOW — must match original paper run
# ============================================================
N_PAPER          = 2000
MARGIN_THR       = 0.0338
WARN_DEFICIT_THR = 0.03

BASE_POLICY = dict(
    w_sensor=0.20,
    w_window=0.80,
    warn_threshold=0.50,
    failure_threshold=0.50,
    drift_threshold=0.35,
)

# ============================================================
# LOAD — labels and PRECOMP only
# ============================================================
print("Loading labels...")
y            = np.load(label_path)
holdout_mask = np.load(holdout_mask_path).astype(bool)
y_hold       = y[holdout_mask]
print(f"✅ y_hold shape: {y_hold.shape}")

print("Loading PRECOMP (~30 seconds)...")
with open(PRECOMP_PATH, "r") as f:
    PRECOMP = json.load(f)
print(f"✅ PRECOMP loaded: {len(PRECOMP)} total samples")

# Slice to paper evaluation window
PRECOMP_2k = PRECOMP[:N_PAPER]
y_hold_2k  = y_hold[:N_PAPER]
print(f"✅ Using first {N_PAPER} samples — matching paper evaluation")

# Quick sanity check — confirm y_true in PRECOMP matches y_hold
mismatch = sum(
    1 for i, row in enumerate(PRECOMP_2k)
    if int(row["y_true"]) != int(y_hold_2k[i])
)
if mismatch == 0:
    print("✅ PRECOMP y_true matches y_hold — data aligned correctly")
else:
    print(f"⚠️ WARNING: {mismatch} y_true mismatches — check data alignment")

# ============================================================
# DECISION AGENT — minimal self-contained version
# ============================================================

class DecisionAgent:

    def __init__(
        self,
        w_sensor: float = 0.20,
        w_window: float = 0.80,
        w_wdi: float = 0.60,
        w_sensor_drift: float = 0.30,
        w_fdi: float = 0.10,
        failure_threshold: float = 0.50,
        failure_critical_threshold: float = 0.80,
        detection_threshold: float = 0.50,
        drift_threshold: float = 0.35,
        warn_threshold: float = 0.50,
        warn_gate_high_contra: float = 0.65,
        warn_boost: float = 0.15,
        warn_penalty: float = 0.25,
        alert_low: float = 0.35,
        alert_med: float = 0.55,
        alert_high: float = 0.75,
        topk_sensors: int = 5,
        use_confidence: bool = True,
    ):
        self.w_sensor                  = w_sensor
        self.w_window                  = w_window
        self.w_wdi                     = w_wdi
        self.w_sensor_drift            = w_sensor_drift
        self.w_fdi                     = w_fdi
        self.failure_threshold         = failure_threshold
        self.failure_critical_threshold = failure_critical_threshold
        self.detection_threshold       = detection_threshold
        self.drift_threshold           = drift_threshold
        self.warn_threshold            = warn_threshold
        self.warn_gate_high_contra     = warn_gate_high_contra
        self.warn_boost                = warn_boost
        self.warn_penalty              = warn_penalty
        self.alert_low                 = alert_low
        self.alert_med                 = alert_med
        self.alert_high                = alert_high
        self.topk_sensors              = topk_sensors
        self.use_confidence            = use_confidence

    def _clip01(self, x):
        try:
            return float(np.clip(float(x), 0.0, 1.0))
        except Exception:
            return 0.0

    def _sensor_intensity(self, master_output):
        if not master_output:
            return {
                "sensor_anom_intensity":   0.0,
                "sensor_drift_intensity":  0.0,
                "sensor_retrain_intensity": 0.0,
                "top_sensors": [],
                "rates": {
                    "anomaly_rate": 0.0,
                    "drift_rate":   0.0,
                    "retrain_rate": 0.0,
                }
            }
        sys_dec = master_output.get("system_decisions", {}) or {}
        rates = {
            "anomaly_rate": float(sys_dec.get("anomaly_rate", 0.0)),
            "drift_rate":   float(sys_dec.get("drift_rate",   0.0)),
            "retrain_rate": float(sys_dec.get("retrain_rate", 0.0)),
        }
        sensor_results = master_output.get("sensor_results", []) or []
        strengths = []
        for r in sensor_results:
            score  = float(r.get("anomaly_score", 0.0))
            conf   = float(r.get("confidence", 0.5))
            is_an  = 1.0 if r.get("is_anomaly", False) else 0.0
            dr     = 1.0 if r.get("drift_flag", False) else 0.0
            rt     = 1.0 if r.get("needs_retrain_flag", False) else 0.0
            base   = score * (conf if self.use_confidence else 1.0)
            strengths.append({
                "strength":          float(base + 0.25 * is_an),
                "drift_flag":        bool(dr),
                "needs_retrain_flag": bool(rt),
            })
        strengths_sorted = sorted(
            strengths, key=lambda x: x["strength"], reverse=True
        )
        top  = strengths_sorted[:max(1, self.topk_sensors)]
        vals = np.array([t["strength"] for t in top], dtype=float)
        top_strength_mean = float(
            1.0 - np.exp(-np.mean(np.maximum(vals, 0.0)))
        ) if len(vals) else 0.0
        drift_flags   = np.array(
            [1.0 if t["drift_flag"] else 0.0 for t in strengths_sorted],
            dtype=float
        )
        retrain_flags = np.array(
            [1.0 if t["needs_retrain_flag"] else 0.0 for t in strengths_sorted],
            dtype=float
        )
        return {
            "sensor_anom_intensity": self._clip01(
                0.5 * rates["anomaly_rate"] + 0.5 * top_strength_mean),
            "sensor_drift_intensity": self._clip01(
                0.6 * rates["drift_rate"] +
                0.4 * float(drift_flags.mean() if len(drift_flags) else 0.0)),
            "sensor_retrain_intensity": self._clip01(
                0.6 * rates["retrain_rate"] +
                0.4 * float(retrain_flags.mean() if len(retrain_flags) else 0.0)),
            "top_sensors": top,
            "rates": rates,
        }

    def _window_intensity(self, window_output):
        if not window_output:
            return {
                "win_anom_intensity":  0.0,
                "win_drift_intensity": 0.0,
                "fds": 0.0, "fdi": 0.0, "wss": 0.0, "wdi": 0.0,
                "event_type": None, "severity": 0.0,
                "window_mse": None, "predicted_window": None,
            }
        event_type = window_output.get("event_type")
        severity   = float(window_output.get("severity", 0.0) or 0.0)
        fds = float(window_output.get("fds", 0.0) or 0.0)
        fdi = float(window_output.get("fdi", 0.0) or 0.0)
        wss = float(window_output.get("wss", 0.0) or 0.0)
        wdi = float(window_output.get("wdi", 0.0) or 0.0)
        window_mse = None
        if window_output.get("forecast_metrics"):
            window_mse = window_output["forecast_metrics"].get("mse")
        wss_int = 1.0 / (1.0 + np.exp(-abs(wss)))
        fds_int = 1.0 / (1.0 + np.exp(-fds))
        sev_int = 1.0 - np.exp(-max(severity, 0.0))
        win_anom = self._clip01(
            0.85 * wss_int + 0.10 * sev_int + 0.05 * fds_int +
            (0.05 if event_type == "ANOMALY" else 0.0)
        )
        win_drift = self._clip01(
            0.85 * wdi + 0.15 * fdi +
            (0.10 if event_type == "DRIFT" else 0.0)
        )
        return {
            "win_anom_intensity":  float(win_anom),
            "win_drift_intensity": float(win_drift),
            "fds": fds, "fdi": fdi, "wss": wss, "wdi": wdi,
            "event_type":      event_type,
            "severity":        severity,
            "window_mse":      float(window_mse) if window_mse is not None else None,
            "predicted_window": window_output.get("predicted_window"),
        }

    def _prediction_probs(self, model_outputs):
        if not model_outputs:
            return {"p_fault": 0.0, "p_warn": 0.0, "p_normal": 1.0}
        p_fault = float(model_outputs.get("failure_prob",     0.0) or 0.0)
        p_warn  = float(model_outputs.get("transformer_prob", 0.0) or 0.0)
        p_norm  = float(model_outputs.get(
            "p_normal", max(0.0, 1.0 - p_fault - p_warn)))
        return {
            "p_fault":  self._clip01(p_fault),
            "p_warn":   self._clip01(p_warn),
            "p_normal": self._clip01(p_norm),
        }

    def decide(self, master_output, window_output,
               model_outputs=None, metadata=None):

        sens = self._sensor_intensity(master_output)
        win  = self._window_intensity(window_output)
        pred = self._prediction_probs(model_outputs)

        detection_risk = self._clip01(
            self.w_sensor * sens["sensor_anom_intensity"] +
            self.w_window * win["win_anom_intensity"]
        )
        drift_risk = self._clip01(
            max(win["wdi"], sens["sensor_drift_intensity"])
        )

        final_failure = (
            pred["p_fault"] >= self.failure_threshold
            or (
                pred["p_warn"] >= self.warn_threshold
                and detection_risk >= 0.65
                and drift_risk    >= 0.60
            )
        )

        final_anomaly = detection_risk >= self.detection_threshold
        final_drift   = drift_risk     >= self.drift_threshold

        arg         = (model_outputs.get("transformer_argmax")
                       if model_outputs else None)
        warn_margin = float(pred["p_warn"] - pred["p_normal"])

        if arg == 1:
            if warn_margin >= 0.06:
                warning_level = "HIGH"
            elif warn_margin >= 0.04:
                warning_level = "MEDIUM"
            else:
                warning_level = "LOW"
        else:
            warning_level = None

        margin_gate         = warn_margin >= MARGIN_THR
        transformer_warning = (arg == 1) and margin_gate
        near_miss           = (arg == 0) and (
            0 < (pred["p_normal"] - pred["p_warn"]) < WARN_DEFICIT_THR
        )
        warning_evidence    = near_miss and (
            drift_risk     >= self.drift_threshold or
            detection_risk >= self.detection_threshold
        )

        final_warning = (
            (transformer_warning or warning_evidence)
            and not final_failure
        )

        return {
            "final_warning": bool(final_warning),
            "final_failure": bool(final_failure),
            "final_anomaly": bool(final_anomaly),
            "final_drift":   bool(final_drift),
            "scores": {
                "p_warn":          pred["p_warn"],
                "p_fault":         pred["p_fault"],
                "p_normal":        pred["p_normal"],
                "warn_margin":     float(warn_margin),
                "warning_level":   warning_level,
                "detection_risk":  float(detection_risk),
                "drift_risk":      float(drift_risk),
            },
        }


# ============================================================
# ABLATION RUNNER
# ============================================================

def run_ablation_variant(precomp, y_hold, variant_name,
                          use_sensor=True,
                          use_window=True,
                          margin_thr=MARGIN_THR,
                          warn_deficit=WARN_DEFICIT_THR,
                          policy=None):
    if policy is None:
        policy = BASE_POLICY.copy()

    agent       = DecisionAgent(**policy)
    dec_warn    = []
    y_warn_true = []
    y_fail_true = []

    for i, row in enumerate(precomp):
        y_true = int(row["y_true"])
        mo     = row.get("model_outputs", {}) or {}

        p0     = float(mo.get("p_normal", 0.0))
        p1     = float(mo.get("p_warn",   0.0))
        p2     = float(mo.get("p_fault",  0.0))
        argmax = int(mo.get("argmax", np.argmax([p0, p1, p2])))

        master_out = row["master"] if use_sensor else None
        window_out = row["window"] if use_window else None

        decision = agent.decide(
            master_output=master_out,
            window_output=window_out,
            model_outputs={
                "failure_prob":       p2,
                "transformer_prob":   p1,
                "transformer_argmax": argmax,
                "p_normal":           p0,
            },
            metadata={"index": i}
        )

        warn_margin = p1 - p0

        if not use_sensor and not use_window:
            # V6: post-processing only — margin gate, no context
            warn = int((argmax == 1) and (warn_margin >= margin_thr))

        elif margin_thr == 0.0:
            # V4: no margin gate
            warn = int((argmax == 1) and not decision["final_failure"])

        elif warn_deficit == 0.0:
            # V5: no near-miss recovery
            warn = int(
                (argmax == 1)
                and (warn_margin >= margin_thr)
                and not decision["final_failure"]
            )

        else:
            # V1, V2, V3: full decision agent output
            warn = int(decision["final_warning"])

        dec_warn.append(warn)
        y_warn_true.append(int(y_true == 1))
        y_fail_true.append(int(y_true == 2))

    y_w = np.array(y_warn_true)
    d_w = np.array(dec_warn)

    return {
        "variant":           variant_name,
        "warning_precision": float(precision_score(y_w, d_w, zero_division=0)),
        "warning_recall":    float(recall_score(y_w,    d_w, zero_division=0)),
        "warning_f1":        float(f1_score(y_w,        d_w, zero_division=0)),
        "n_samples":         len(y_w),
    }


# ============================================================
# V0 — Transformer argmax baseline
# Derived directly from PRECOMP, not from state
# ============================================================
y_warn_2k = np.array([int(r["y_true"] == 1) for r in PRECOMP_2k])
y_fail_2k = np.array([int(r["y_true"] == 2) for r in PRECOMP_2k])
tf_warn_2k = np.array([
    int(r["model_outputs"]["argmax"] == 1) for r in PRECOMP_2k
])

v0 = {
    "variant":           "V0: Transformer argmax (baseline)",
    "warning_precision": float(precision_score(y_warn_2k, tf_warn_2k,
                                               zero_division=0)),
    "warning_recall":    float(recall_score(y_warn_2k,    tf_warn_2k,
                                            zero_division=0)),
    "warning_f1":        float(f1_score(y_warn_2k,        tf_warn_2k,
                                        zero_division=0)),
    "n_samples":         N_PAPER,
}
print(f"✅ V0: P={v0['warning_precision']:.3f}  "
      f"R={v0['warning_recall']:.3f}  F1={v0['warning_f1']:.3f}")
print(f"   (expected: P=0.875  R=0.941  F1=0.906)")


# ============================================================
# V1–V6 — all variants
# ============================================================
print("\nRunning ablation variants (each ~1 min)...")

v1 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V1: Full ASPIRE (reported result)")
print(f"✅ V1: P={v1['warning_precision']:.3f}  "
      f"R={v1['warning_recall']:.3f}  F1={v1['warning_f1']:.3f}")
print(f"   (expected: P=0.889  R=0.952  F1=0.919)")

v2 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V2: No sensor agents",
    use_sensor=False)
print(f"✅ V2 done: P={v2['warning_precision']:.3f}  "
      f"R={v2['warning_recall']:.3f}  F1={v2['warning_f1']:.3f}")

v3 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V3: No window agent",
    use_window=False)
print(f"✅ V3 done: P={v3['warning_precision']:.3f}  "
      f"R={v3['warning_recall']:.3f}  F1={v3['warning_f1']:.3f}")

v4 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V4: No margin gate",
    margin_thr=0.0)
print(f"✅ V4 done: P={v4['warning_precision']:.3f}  "
      f"R={v4['warning_recall']:.3f}  F1={v4['warning_f1']:.3f}")

v5 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V5: No near-miss recovery",
    warn_deficit=0.0)
print(f"✅ V5 done: P={v5['warning_precision']:.3f}  "
      f"R={v5['warning_recall']:.3f}  F1={v5['warning_f1']:.3f}")

v6 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V6: Post-proc baseline (margin gate only)",
    use_sensor=False,
    use_window=False,
    margin_thr=MARGIN_THR)
print(f"✅ V6 done: P={v6['warning_precision']:.3f}  "
      f"R={v6['warning_recall']:.3f}  F1={v6['warning_f1']:.3f}")


# ============================================================
# RESULTS TABLE
# ============================================================
results = [v0, v1, v2, v3, v4, v5, v6]

print(f"\n{'Variant':<50} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 80)
for r in results:
    marker = "  ◀ reported" if "V1" in r["variant"] else ""
    print(f"{r['variant']:<50} "
          f"{r['warning_precision']:>10.3f} "
          f"{r['warning_recall']:>8.3f} "
          f"{r['warning_f1']:>8.3f}{marker}")


# ============================================================
# SAVE TO DRIVE
# ============================================================
with open(ABLATION_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Saved to {ABLATION_PATH}")




# ============================================================
# SENSITIVITY ANALYSIS — A7 REVISED
# Two meaningful sweeps + retain flat results for honest reporting
# MARGIN_THR already covered by Fig 4 — cited in writing only
# ============================================================

from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import json

SENSITIVITY_PATH2 = "/content/drive/MyDrive/PHD/2025/sensitivity_results_v2.json"
SENSITIVITY_FIG2  = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_v2.pdf"
SENSITIVITY_PNG2  = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_v2.png"


# ============================================================
# SENSITIVITY ANALYSIS — A7 REVISED
# Two meaningful sweeps + retain flat results for honest reporting
# MARGIN_THR already covered by Fig 4 — cited in writing only
# ============================================================

from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import json

SENSITIVITY_PATH2 = "/content/drive/MyDrive/PHD/2025/sensitivity_results_v2.json"
SENSITIVITY_FIG2  = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_v2.pdf"
SENSITIVITY_PNG2  = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_v2.png"


# ============================================================
# RUNNER A — sweep WARN_DEFICIT_THR
# Controls size of near-miss recovery window
# All other params at calibrated values
# ============================================================

def run_deficit_sweep(precomp, warn_deficit):
    """Sweep WARN_DEFICIT_THR, all other params fixed."""
    policy = BASE_POLICY.copy()
    agent  = DecisionAgent(**policy)

    dec_warn    = []
    y_warn_true = []

    for row in precomp:
        y_true = int(row["y_true"])
        mo     = row.get("model_outputs", {}) or {}
        p0     = float(mo.get("p_normal", 0.0))
        p1     = float(mo.get("p_warn",   0.0))
        p2     = float(mo.get("p_fault",  0.0))
        argmax = int(mo.get("argmax", np.argmax([p0, p1, p2])))

        warn_margin_val    = p1 - p0
        margin_gate_val    = warn_margin_val >= MARGIN_THR
        transformer_warn_v = (argmax == 1) and margin_gate_val

        # Near-miss uses the SWEPT deficit value
        near_miss_val = (argmax == 0) and (
            0 < (p0 - p1) < warn_deficit
        )

        decision = agent.decide(
            master_output=row["master"],
            window_output=row["window"],
            model_outputs={
                "failure_prob":       p2,
                "transformer_prob":   p1,
                "transformer_argmax": argmax,
                "p_normal":           p0,
            }
        )

        warning_evidence_v = near_miss_val and (
            decision["scores"]["drift_risk"]     >= 0.35 or
            decision["scores"]["detection_risk"] >= 0.50
        )
        warn = int(
            (transformer_warn_v or warning_evidence_v)
            and not decision["final_failure"]
        )

        dec_warn.append(warn)
        y_warn_true.append(int(y_true == 1))

    y_w = np.array(y_warn_true)
    d_w = np.array(dec_warn)

    return (
        float(precision_score(y_w, d_w, zero_division=0)),
        float(recall_score(y_w,    d_w, zero_division=0)),
        float(f1_score(y_w,        d_w, zero_division=0)),
    )


# ============================================================
# RUNNER B — sweep failure_threshold (τF)
# Controls when fault predictions override warning decisions
# All other params at calibrated values
# ============================================================

def run_failure_thr_sweep(precomp, failure_thr):
    """Sweep failure_threshold, all other params fixed."""
    policy = BASE_POLICY.copy()
    policy["failure_threshold"] = failure_thr
    agent  = DecisionAgent(**policy)

    dec_warn    = []
    y_warn_true = []

    for row in precomp:
        y_true = int(row["y_true"])
        mo     = row.get("model_outputs", {}) or {}
        p0     = float(mo.get("p_normal", 0.0))
        p1     = float(mo.get("p_warn",   0.0))
        p2     = float(mo.get("p_fault",  0.0))
        argmax = int(mo.get("argmax", np.argmax([p0, p1, p2])))

        warn_margin_val    = p1 - p0
        margin_gate_val    = warn_margin_val >= MARGIN_THR
        transformer_warn_v = (argmax == 1) and margin_gate_val
        near_miss_val      = (argmax == 0) and (
            0 < (p0 - p1) < WARN_DEFICIT_THR
        )

        decision = agent.decide(
            master_output=row["master"],
            window_output=row["window"],
            model_outputs={
                "failure_prob":       p2,
                "transformer_prob":   p1,
                "transformer_argmax": argmax,
                "p_normal":           p0,
            }
        )

        warning_evidence_v = near_miss_val and (
            decision["scores"]["drift_risk"]     >= 0.35 or
            decision["scores"]["detection_risk"] >= 0.50
        )

        # final_failure now uses swept threshold
        final_failure_v = (
            p2 >= failure_thr
            or (
                p1 >= 0.50
                and decision["scores"]["detection_risk"] >= 0.65
                and decision["scores"]["drift_risk"]     >= 0.60
            )
        )

        warn = int(
            (transformer_warn_v or warning_evidence_v)
            and not final_failure_v
        )

        dec_warn.append(warn)
        y_warn_true.append(int(y_true == 1))

    y_w = np.array(y_warn_true)
    d_w = np.array(dec_warn)

    return (
        float(precision_score(y_w, d_w, zero_division=0)),
        float(recall_score(y_w,    d_w, zero_division=0)),
        float(f1_score(y_w,        d_w, zero_division=0)),
    )


# ============================================================
# PARAMETER RANGES
# ============================================================

# WARN_DEFICIT_THR sweep (calibrated = 0.03)
# Range 0.00 (no near-miss recovery) to 0.15 (wide recovery window)
deficit_values = np.round(np.arange(0.00, 0.16, 0.01), 2).tolist()

# failure_threshold sweep (calibrated = 0.50)
# Range 0.25 (very sensitive) to 0.80 (very conservative)
failure_thr_values = np.round(np.arange(0.25, 0.81, 0.05), 2).tolist()


# ============================================================
# RUN SWEEPS
# ============================================================

print("Sweeping WARN_DEFICIT_THR (calibrated=0.03)...")
deficit_results = []
for v in deficit_values:
    p, r, f = run_deficit_sweep(PRECOMP_2k, warn_deficit=v)
    deficit_results.append({
        "value": float(v), "precision": p, "recall": r, "f1": f
    })
    marker = " ◀ calibrated" if abs(v - 0.03) < 0.005 else ""
    print(f"  deficit={v:.2f}: P={p:.3f} R={r:.3f} F1={f:.3f}{marker}")

print("\nSweeping failure_threshold τF (calibrated=0.50)...")
failure_thr_results = []
for v in failure_thr_values:
    p, r, f = run_failure_thr_sweep(PRECOMP_2k, failure_thr=v)
    failure_thr_results.append({
        "value": float(v), "precision": p, "recall": r, "f1": f
    })
    marker = " ◀ calibrated" if abs(v - 0.50) < 0.03 else ""
    print(f"  failure_thr={v:.2f}: P={p:.3f} R={r:.3f} F1={f:.3f}{marker}")


# ============================================================
# SAVE
# ============================================================

sensitivity_data_v2 = {
    "note": (
        "MARGIN_THR sweep reported as Fig 4. "
        "drift_threshold, detection_threshold, w_window sweeps "
        "confirmed flat — reported as robustness finding. "
        "This file contains the two meaningful variation sweeps."
    ),
    "warn_deficit":      deficit_results,
    "failure_threshold": failure_thr_results,
    "flat_sweeps_summary": {
        "drift_threshold":     "F1=0.920 constant across 0.15-0.60",
        "detection_threshold": "F1=0.920 constant across 0.20-0.75, 0.919 at 0.80",
        "w_window":            "F1=0.920 constant across 0.50-1.00, 0.919 at 0.30-0.40",
    },
    "calibrated_values": {
        "MARGIN_THR":          MARGIN_THR,
        "WARN_DEFICIT_THR":    WARN_DEFICIT_THR,
        "failure_threshold":   0.50,
        "drift_threshold":     0.35,
        "detection_threshold": 0.50,
        "w_window":            0.80,
    }
}

with open(SENSITIVITY_PATH2, "w") as f:
    json.dump(sensitivity_data_v2, f, indent=2)
print(f"\n✅ Saved to {SENSITIVITY_PATH2}")


# ============================================================
# PLOT — FLAT SENSITIVITY FIGURE (publication quality)
# Three robust parameters — described as robustness finding
# ============================================================

SENSITIVITY_FLAT_PDF = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_flat_pub.pdf"
SENSITIVITY_FLAT_PNG = "/content/drive/MyDrive/PHD/2025/sensitivity_analysis_flat_pub.png"

# Load flat results from previous run
# (drift_results, detection_results, weight_results already in memory
#  from the first sensitivity cell — if not, reload from JSON)
with open("/content/drive/MyDrive/PHD/2025/sensitivity_results.json") as f:
    flat_data = json.load(f)

drift_results     = flat_data["drift_threshold"]
detection_results = flat_data["detection_threshold"]
weight_results    = flat_data["w_window"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_flat(ax, results, xlabel, calibrated_val, key="value"):
    xs  = [r[key]         for r in results]
    p   = [r["precision"] for r in results]
    r_  = [r["recall"]    for r in results]
    f1  = [r["f1"]        for r in results]

    ax.plot(xs, p,  label="Precision", linewidth=2.5,
            marker="o", markersize=5)
    ax.plot(xs, r_, label="Recall",    linewidth=2.5,
            marker="s", markersize=5)
    ax.plot(xs, f1, label="F1",        linewidth=2.5,
            linestyle="--", marker="^", markersize=5)
    ax.axvline(calibrated_val, color="black", linestyle=":",
               linewidth=2.0, label="Calibrated value")
    ax.set_xlabel(xlabel,  fontsize=14)
    ax.set_ylabel("Score", fontsize=14)
    ax.set_ylim(0.85, 1.00)
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=11, loc="center right")

plot_flat(
    axes[0], drift_results,
    r"$\tau_{drift}$ (drift risk threshold)",
    calibrated_val=0.35
)

plot_flat(
    axes[1], detection_results,
    r"$\tau_{det}$ (detection risk threshold)",
    calibrated_val=0.50
)

plot_flat(
    axes[2], weight_results,
    r"$w_{window}$ (window agent weight)",
    calibrated_val=0.80,
    key="w_window"
)

plt.tight_layout(pad=2.0)
fig.savefig(SENSITIVITY_FLAT_PDF, dpi=300, bbox_inches="tight")
fig.savefig(SENSITIVITY_FLAT_PNG, dpi=300, bbox_inches="tight")
plt.show()
print("✅ Flat sensitivity figure saved")

# ============================================================
# SENSITIVITY ANALYSIS — COMBINED PUBLICATION FIGURE v4
# Fixed layout + black fonts throughout
# ============================================================

SENSITIVITY_COMBINED_PDF = "/content/drive/MyDrive/PHD/2025/sensitivity_combined_v4.pdf"
SENSITIVITY_COMBINED_PNG = "/content/drive/MyDrive/PHD/2025/sensitivity_combined_v4.png"

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import numpy as np
import json

# Reset first to clear any lingering state
mpl.rcParams.update(mpl.rcParamsDefault)

# Global font settings — all black
mpl.rcParams.update({
    "font.size":         14,
    "axes.titlesize":    15,
    "axes.labelsize":    14,
    "xtick.labelsize":   13,
    "ytick.labelsize":   13,
    "legend.fontsize":   12,
    "text.color":        "black",
    "axes.labelcolor":   "black",
    "xtick.color":       "black",
    "ytick.color":       "black",
    "axes.edgecolor":    "black",
})

# Load flat results
with open("/content/drive/MyDrive/PHD/2025/sensitivity_results.json") as f:
    flat_data = json.load(f)

drift_results     = flat_data["drift_threshold"]
detection_results = flat_data["detection_threshold"]
weight_results    = flat_data["w_window"]

# ============================================================
# BUILD FIGURE WITH EXPLICIT SUBPLOT POSITIONS
# Use subplot2grid to avoid GridSpec row confusion
# ============================================================

fig = plt.figure(figsize=(20, 11))

# Row 1 — three flat panels
ax1 = plt.subplot2grid((2, 6), (0, 0), colspan=2, fig=fig)
ax2 = plt.subplot2grid((2, 6), (0, 2), colspan=2, fig=fig)
ax3 = plt.subplot2grid((2, 6), (0, 4), colspan=2, fig=fig)

# Row 2 — two varying panels (wider, centred)
ax4 = plt.subplot2grid((2, 6), (1, 0), colspan=3, fig=fig)
ax5 = plt.subplot2grid((2, 6), (1, 3), colspan=3, fig=fig)

plt.subplots_adjust(
    hspace=0.55,
    wspace=0.40,
    top=0.93,
    bottom=0.08
)

# ============================================================
# SHARED PLOT FUNCTION
# ============================================================

def plot_panel(ax, results, xlabel, calibrated_val,
               key="value", ylim=(0.84, 1.00),
               show_ylabel=True, panel_label=None):

    xs  = [r[key]         for r in results]
    p   = [r["precision"] for r in results]
    r_  = [r["recall"]    for r in results]
    f1  = [r["f1"]        for r in results]

    ax.plot(xs, p,  label="Precision",
            linewidth=2.5, marker="o",
            markersize=6, markevery=2)
    ax.plot(xs, r_, label="Recall",
            linewidth=2.5, marker="s",
            markersize=6, markevery=2)
    ax.plot(xs, f1, label="F1",
            linewidth=2.5, linestyle="--",
            marker="^", markersize=6, markevery=2)
    ax.axvline(calibrated_val, color="black",
               linestyle=":", linewidth=2.2,
               label="Calibrated")

    ax.set_xlabel(xlabel, fontsize=14,
                  labelpad=8, color="black")

    if show_ylabel:
        ax.set_ylabel("Score", fontsize=14,
                      labelpad=8, color="black")

    ax.set_ylim(ylim)
    ax.tick_params(axis="both", labelsize=13,
                   pad=5, colors="black")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=12, loc="best",
              framealpha=0.8,
              handlelength=2.0,
              borderpad=0.6,
              labelcolor="black")

    if panel_label:
        ax.set_title(panel_label, fontsize=15,
                     fontweight="bold", loc="left",
                     pad=8, color="black")

# ============================================================
# ROW 1 — FLAT / ROBUST PARAMETERS
# ============================================================

plot_panel(
    ax1, drift_results,
    r"$\tau_{drift}$ (drift risk threshold)",
    calibrated_val=0.35,
    ylim=(0.84, 1.00),
    show_ylabel=True,
    panel_label="(a)"
)

plot_panel(
    ax2, detection_results,
    r"$\tau_{det}$ (detection risk threshold)",
    calibrated_val=0.50,
    ylim=(0.84, 1.00),
    show_ylabel=False,
    panel_label="(b)"
)

plot_panel(
    ax3, weight_results,
    r"$w_{window}$ (window agent weight)",
    calibrated_val=0.80,
    ylim=(0.84, 1.00),
    show_ylabel=False,
    key="w_window",
    panel_label="(c)"
)

# ============================================================
# ROW 2 — MEANINGFUL VARIATION PARAMETERS
# ============================================================

plot_panel(
    ax4, deficit_results,
    r"$\Delta_{deficit}$ (near-miss tolerance)",
    calibrated_val=0.03,
    ylim=(0.70, 1.01),
    show_ylabel=True,
    panel_label="(d)"
)

plot_panel(
    ax5, failure_thr_results,
    r"$\tau_{F}$ (failure probability threshold)",
    calibrated_val=0.50,
    ylim=(0.70, 1.01),
    show_ylabel=False,
    panel_label="(e)"
)

# ============================================================
# ROW HEADER ANNOTATIONS — black, italic
# Positioned relative to figure coordinates
# ============================================================

fig.text(
    0.5, 0.955,
    "Robust parameters — F1 constant across swept range",
    ha="center", va="bottom",
    fontsize=13, style="italic",
    color="black",
    fontweight="normal"
)

fig.text(
    0.5, 0.475,
    "Calibration-sensitive parameters — meaningful variation",
    ha="center", va="bottom",
    fontsize=13, style="italic",
    color="black",
    fontweight="normal"
)

# ============================================================
# SAVE
# ============================================================

fig.savefig(SENSITIVITY_COMBINED_PDF, dpi=300, bbox_inches="tight")
fig.savefig(SENSITIVITY_COMBINED_PNG, dpi=300, bbox_inches="tight")
plt.show()

# Reset rcParams
mpl.rcParams.update(mpl.rcParamsDefault)

print(f"✅ Saved:")
print(f"   PDF: {SENSITIVITY_COMBINED_PDF}")
print(f"   PNG: {SENSITIVITY_COMBINED_PNG}")

# ============================================================
# COST-EFFECTIVENESS ANALYSIS — Hughes et al. (2025) framework
# Applied to ASPIRE early-warning prediction task
# All values derived from existing TP/FP/FN results
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

COST_FIG_PDF = "/content/drive/MyDrive/PHD/2025/cer_analysis.pdf"
COST_FIG_PNG = "/content/drive/MyDrive/PHD/2025/cer_analysis.png"
COST_JSON    = "/content/drive/MyDrive/PHD/2025/cer_results.json"

# ============================================================
# KNOWN VALUES FROM EVALUATION RESULTS
# ============================================================

# Ground truth warnings in 2000-window holdout
N_warn_true = 726

# Transformer baseline
tf_P  = 0.875
tf_R  = 0.941
tf_TP = round(N_warn_true * tf_R)   # 683
tf_FP = round(tf_TP / tf_P - tf_TP) # 98
tf_FN = N_warn_true - tf_TP          # 43

# ASPIRE
asp_P  = 0.889
asp_R  = 0.952
asp_TP = round(N_warn_true * asp_R)   # 691
asp_FP = round(asp_TP / asp_P - asp_TP) # 86
asp_FN = N_warn_true - asp_TP           # 35

# Falsehood ratio FR = FP / (FN + TP)
tf_FR  = tf_FP  / (tf_FN  + tf_TP)
asp_FR = asp_FP / (asp_FN + asp_TP)

print("=== INPUT VALUES ===")
print(f"Transformer: TP={tf_TP}  FP={tf_FP}  FN={tf_FN}")
print(f"ASPIRE:      TP={asp_TP} FP={asp_FP} FN={asp_FN}")
print(f"Transformer FR: {tf_FR:.4f}")
print(f"ASPIRE FR:      {asp_FR:.4f}")
print(f"Transformer Precision: {tf_P}")
print(f"ASPIRE Precision:      {asp_P}")


# ============================================================
# CER FUNCTION — Hughes et al. Equation 5
# CER = (Recall - FMCR * FR) / (1 - FMCR)
# Extended with alpha:
# CER_alpha = (Recall - alpha * FMCR * (Recall + FR)) / (1 - FMCR)
# ============================================================

def cer(recall, fr, fmcr, alpha=1.0):
    numerator   = recall - alpha * fmcr * (recall + fr)
    denominator = 1 - fmcr
    return numerator / denominator


# ============================================================
# SWEEP FMCR
# ============================================================

fmcr_values = np.linspace(0.05, 0.80, 300)

# Alpha values
alpha_no_expert = 1.00   # no explanation
alpha_expert    = 0.75   # Expert Agent reduces FP response cost by 25%
                         # — illustrative, stated as such in paper

# Transformer (no Expert Agent — baseline has no explanation)
tf_cer = [cer(tf_R, tf_FR, f, alpha=1.0) for f in fmcr_values]

# ASPIRE without Expert Agent
asp_cer = [cer(asp_R, asp_FR, f, alpha=1.0) for f in fmcr_values]

# ASPIRE with Expert Agent
asp_cer_expert = [cer(asp_R, asp_FR, f, alpha=alpha_expert)
                  for f in fmcr_values]

# Cost-efficiency thresholds
tf_threshold  = tf_P   # FMCR must be below this
asp_threshold = asp_P  # FMCR must be below this


# ============================================================
# PRINT KEY VALUES AT EXPECTED OPERATIONAL RANGE
# ============================================================

print("\n=== CER AT KEY FMCR VALUES ===")
for fmcr_check in [0.20, 0.25, 0.33, 0.50]:
    tf_v      = cer(tf_R,  tf_FR,  fmcr_check, alpha=1.0)
    asp_v     = cer(asp_R, asp_FR, fmcr_check, alpha=1.0)
    asp_exp_v = cer(asp_R, asp_FR, fmcr_check, alpha=alpha_expert)
    print(f"FMCR={fmcr_check:.2f}: "
          f"Transformer={tf_v:.3f}  "
          f"ASPIRE={asp_v:.3f}  "
          f"ASPIRE+Expert={asp_exp_v:.3f}  "
          f"Gain={asp_v-tf_v:.3f}")

print(f"\nCost-efficiency threshold:")
print(f"  Transformer: FMCR < {tf_threshold:.3f}")
print(f"  ASPIRE:      FMCR < {asp_threshold:.3f}")

# Utility gain
delta_TP = asp_TP - tf_TP
delta_FP = tf_FP - asp_FP
print(f"\nUtility gain components:")
print(f"  Additional TPs: {delta_TP}")
print(f"  Fewer FPs:      {delta_FP}")
print(f"  DeltaU = {delta_TP} * C_FN + {delta_FP} * C_FP")
print(f"  (positive for all positive cost values)")


# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Panel 1: CER curves ──
ax = axes[0]

ax.plot(fmcr_values, tf_cer,
        label="Transformer baseline",
        linewidth=2.2, linestyle="--", color="#5ba3c9")

ax.plot(fmcr_values, asp_cer,
        label="ASPIRE (no Expert Agent, $\\alpha$=1.0)",
        linewidth=2.2, color="#1F4E79")

ax.plot(fmcr_values, asp_cer_expert,
        label=f"ASPIRE + Expert Agent ($\\alpha$={alpha_expert})",
        linewidth=2.2, color="#E07B39")

# Shade expected operational range
ax.axvspan(0.20, 0.33, alpha=0.10, color="#27ae60",
           label="Expected operational range")

ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_xlabel("FMCR ($C_{FP}$ / $C_{FN}$)", fontsize=13)
ax.set_ylabel("Cost-Effectiveness Ratio (CER)", fontsize=13)
ax.set_title("(a) Cost-Effectiveness Ratio", fontsize=13,
             fontweight="bold", loc="left")
ax.tick_params(labelsize=11)
ax.grid(alpha=0.3)
ax.legend(fontsize=10, loc="upper right")
ax.set_xlim(0.05, 0.80)

# ── Panel 2: Cost-efficiency threshold ──
ax2 = axes[1]

fmcr_range = np.linspace(0, 1, 300)

# Shade cost-efficient regions
ax2.axvspan(0, tf_threshold, alpha=0.12,
            color="#5ba3c9", label="Transformer cost-efficient region")
ax2.axvspan(tf_threshold, asp_threshold, alpha=0.15,
            color="#E07B39",
            label="ASPIRE additional cost-efficient region")

ax2.axvline(tf_threshold, color="#5ba3c9", linewidth=2.2,
            linestyle="--",
            label=f"Transformer threshold (FMCR={tf_threshold})")
ax2.axvline(asp_threshold, color="#1F4E79", linewidth=2.2,
            linestyle="-",
            label=f"ASPIRE threshold (FMCR={asp_threshold})")

# Expected operational range
ax2.axvspan(0.20, 0.33, alpha=0.15, color="#27ae60",
            label="Expected operational range")

ax2.set_xlabel("FMCR ($C_{FP}$ / $C_{FN}$)", fontsize=13)
ax2.set_ylabel("Cost-efficiency condition met", fontsize=13)
ax2.set_title("(b) Cost-efficiency thresholds", fontsize=13,
              fontweight="bold", loc="left")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.tick_params(labelsize=11)
ax2.grid(alpha=0.3)
ax2.legend(fontsize=10, loc="upper right")
ax2.set_yticks([])

plt.tight_layout(pad=2.0)
fig.savefig(COST_FIG_PDF, dpi=300, bbox_inches="tight")
fig.savefig(COST_FIG_PNG, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved")


# ============================================================
# SAVE RESULTS
# ============================================================

results = {
    "transformer": {
        "TP": int(tf_TP), "FP": int(tf_FP), "FN": int(tf_FN),
        "precision": tf_P, "recall": tf_R,
        "FR": float(tf_FR),
        "cost_efficiency_threshold": float(tf_threshold),
        "cer_at_fmcr_020": float(cer(tf_R, tf_FR, 0.20)),
        "cer_at_fmcr_033": float(cer(tf_R, tf_FR, 0.33)),
    },
    "aspire": {
        "TP": int(asp_TP), "FP": int(asp_FP), "FN": int(asp_FN),
        "precision": asp_P, "recall": asp_R,
        "FR": float(asp_FR),
        "cost_efficiency_threshold": float(asp_threshold),
        "cer_at_fmcr_020": float(cer(asp_R, asp_FR, 0.20)),
        "cer_at_fmcr_033": float(cer(asp_R, asp_FR, 0.33)),
        "cer_expert_at_fmcr_020": float(
            cer(asp_R, asp_FR, 0.20, alpha=alpha_expert)),
        "cer_expert_at_fmcr_033": float(
            cer(asp_R, asp_FR, 0.33, alpha=alpha_expert)),
    },
    "utility_gain": {
        "delta_TP": int(delta_TP),
        "delta_FP": int(delta_FP),
        "interpretation": (
            f"DeltaU = {delta_TP} * C_FN + {delta_FP} * C_FP, "
            "positive for all positive cost values"
        )
    },
    "alpha_expert": alpha_expert,
    "note": (
        "alpha=0.75 is illustrative — represents 25% reduction "
        "in false-positive response cost through Expert Agent guidance"
    )
}

with open(COST_JSON, "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ Results saved to {COST_JSON}")